# Integración de evidencia, ranking final y conclusión

**Responsable:** Integrante 1
**Proyecto:** Caso 1 · PAH (GRCh38, NM_000277.3, NP_000268.1, PDB 6HYC)

Este notebook reúne todas las capas para las variantes que cambian la proteína, construye el ranking final al lado del inicial y cierra con la conclusión y los límites del análisis.

Entradas:
- `resultados/PAH_pre_vs_Evo2.tsv` (notebook 03: anotación, ranking inicial y Evo2)
- `resultados/rf3/estructura_resumen.tsv` (si todavía no existe, la columna de estructura queda como "Pendiente")

Salidas:
- `resultados/ranking_final.tsv`
- `resultados/tabla_maestra.tsv` (columnas `RF3_or_structure_result`, `final_priority` y `limitations`)


In [1]:
# Configuración: correr esta celda primero
import os
if not os.path.exists("database"):          # en Colab: clonar el repositorio
    if not os.path.exists("Caso_1_PHA_Humano"):
        !git clone https://github.com/XomiDessu/Caso_1_PHA_Humano.git
    %cd Caso_1_PHA_Humano

import pandas as pd
import numpy as np
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)


Clonando en 'Caso_1_PHA_Humano'...
remote: Enumerating objects: 184, done.
remote: Counting objects: 100% (184/184), done.
remote: Compressing objects: 100% (161/161), done.
remote: Total 184 (delta 65), reused 64 (delta 15), pack-reused 0 (from 0)
Recibiendo objetos: 100% (184/184), 3.66 MiB | 6.01 MiB/s, listo.
Resolviendo deltas: 100% (65/65), listo.
/home/geno/Documentos/CUCEI/Genoma/Caso_1_PHA_Humano/notebooks/Caso_1_PHA_Humano


## 1. Cargar la evidencia de las capas anteriores


In [2]:
comp = pd.read_csv("resultados/PAH_pre_vs_Evo2.tsv", sep="\t")

# Solo las variantes que cambian la proteína
prot = comp[comp["consequence"].isin(["missense", "delecion_in_frame"])].copy()

# Estructura (RF3 / 6HYC): se integra si el archivo ya existe
RUTA_RF3 = "resultados/rf3/estructura_resumen.tsv"
if os.path.exists(RUTA_RF3):
    rf3 = pd.read_csv(RUTA_RF3, sep="\t")
    print("Resultados estructurales encontrados:", len(rf3), "filas")
else:
    rf3 = None
    print("Todavía no hay resultados de RF3: la columna de estructura queda como 'Pendiente'.")

print(prot[["ID", "protein_change", "consequence", "dominio", "POPAF", "BLOSUM62",
            "rank_inicial", "rank_evo2", "delta_sitio_genoma", "delta_sitio_gen",
            "delta_ventana_genoma", "delta_ventana_gen"]].sort_values("rank_inicial").to_string(index=False))


Todavía no hay resultados de RF3: la columna de estructura queda como 'Pendiente'.
    ID     protein_change       consequence    dominio    POPAF  BLOSUM62  rank_inicial  rank_evo2  delta_sitio_genoma  delta_sitio_gen  delta_ventana_genoma  delta_ventana_gen
VAR003        p.Arg408Trp          missense catalítico 0.000920      -3.0             1        4.0               -5.72             0.58                -16.06             -31.65
VAR005        p.Asp338Ala          missense catalítico 0.000004      -2.0             2        3.0               -4.14            -0.20                -18.98             -29.51
VAR007        p.Gly247Asp          missense catalítico 0.000010      -1.0             3        1.0               -7.83            -1.48                -22.99             -21.58
VAR011        p.Thr117Ile          missense  regulador 0.000050      -1.0             4        5.0                0.15            -5.18                -11.50              -5.90
VAR012 p.Ala104_Val106del deleci

## 2. Tabla integradora

Para cada variante se resume cada capa y se clasifica la evidencia:

- **Convergente:** prioridad alta en el ranking inicial y fuerte penalización en Evo2.
- **Contradictoria:** las capas, o las medidas de una misma capa, apuntan en direcciones distintas.
- **Insuficiente:** falta una capa para poder evaluarla.
- **Débil:** ninguna capa la señala con fuerza.

La clasificación es una interpretación del equipo basada en los valores de la tabla, no un cálculo automático.


In [3]:
INTERPRETACION = {
    "VAR007": ("Convergente", "Ultra-rara, cambio de carga en el dominio catalítico y primer lugar de Evo2."),
    "VAR003": ("Convergente", "Cambio químico más drástico; Evo2 la penaliza mucho en la ventana del gen (−31.65), pero casi nada en el sitio de esa hebra (+0.58)."),
    "VAR005": ("Convergente", "Ultra-rara, pierde una carga en el dominio catalítico; tercer lugar de Evo2."),
    "VAR014": ("Contradictoria", "Evo2 la penaliza en ambas hebras (−5.94 y −5.85), pero el cambio Glu → Asp es conservador (BLOSUM62 +2)."),
    "VAR011": ("Contradictoria", "Señal de Evo2 dependiente de la hebra (+0.15 genoma, −5.18 gen); a 2 nt del borde de exón, splicing sin evaluar."),
    "VAR015": ("Contradictoria", "Penalizada en la ventana (−13.40) pero casi nada en el sitio de la hebra del gen (−0.03)."),
    "VAR013": ("Débil", "Cambio conservador (BLOSUM62 +3) y penalización moderada en Evo2."),
    "VAR012": ("Insuficiente", "Deleción de 3 aa en el dominio ACT; Evo2 no puede evaluar indels."),
}

def texto_evo2(r):
    if pd.isna(r["rank_evo2"]):
        return "No aplica (deleción)"
    return (f"Lugar {int(r['rank_evo2'])}/14; sitio {r['delta_sitio_genoma']:.2f} / {r['delta_sitio_gen']:.2f}; "
            f"ventana {r['delta_ventana_genoma']:.2f} / {r['delta_ventana_gen']:.2f}")

def texto_estructura(vid):
    if rf3 is None or vid not in set(rf3.iloc[:, 0]):
        return "Pendiente"
    fila = rf3[rf3.iloc[:, 0] == vid].iloc[0]
    return "; ".join(f"{c}: {fila[c]}" for c in rf3.columns[1:])

integradora = pd.DataFrame({
    "Variante": prot["ID"] + " " + prot["protein_change"],
    "Genómica y frecuencia": "POPAF " + prot["POPAF"].map(lambda v: f"{v:g}"),
    "Consecuencia proteica": prot.apply(lambda r: f"{r['consequence']}; {r['dominio']}" +
                                        (f"; BLOSUM62 {int(r['BLOSUM62'])}" if pd.notna(r["BLOSUM62"]) else ""), axis=1),
    "Evo2": prot.apply(texto_evo2, axis=1),
    "Estructura (RF3 / 6HYC)": prot["ID"].map(texto_estructura),
    "Evidencia": prot["ID"].map(lambda v: INTERPRETACION[v][0]),
    "Interpretación": prot["ID"].map(lambda v: INTERPRETACION[v][1]),
    "rank_inicial": prot["rank_inicial"],
}).sort_values("rank_inicial").drop(columns="rank_inicial")

print(integradora.to_string(index=False))


                 Variante Genómica y frecuencia             Consecuencia proteica                                                     Evo2 Estructura (RF3 / 6HYC)      Evidencia                                                                                                                       Interpretación
       VAR003 p.Arg408Trp         POPAF 0.00092 missense; catalítico; BLOSUM62 -3  Lugar 4/14; sitio -5.72 / 0.58; ventana -16.06 / -31.65               Pendiente    Convergente Cambio químico más drástico; Evo2 la penaliza mucho en la ventana del gen (−31.65), pero casi nada en el sitio de esa hebra (+0.58).
       VAR005 p.Asp338Ala           POPAF 4e-06 missense; catalítico; BLOSUM62 -2 Lugar 3/14; sitio -4.14 / -0.20; ventana -18.98 / -29.51               Pendiente    Convergente                                                         Ultra-rara, pierde una carga en el dominio catalítico; tercer lugar de Evo2.
       VAR007 p.Gly247Asp           POPAF 1e-05 missense; catalític

## 3. Ranking final

Como apoyo se calcula el promedio entre el lugar en el ranking inicial y el lugar en Evo2. El README pide que la posición final se justifique con la evidencia y no solo con un puntaje, así que el orden final se decide con ese promedio y con la clasificación de la tabla integradora:

- VAR007, VAR003 y VAR005 (evidencia convergente) ocupan los tres primeros lugares. VAR003 va antes que VAR005 por tener el cambio químico más drástico (BLOSUM62 −3 frente a −2).
- VAR014 y VAR011 empatan en el promedio. VAR014 va antes porque su señal en Evo2 es igual en ambas hebras, mientras que la de VAR011 depende de la hebra.
- VAR012 sale del Top 5 por evidencia insuficiente, no por evidencia en contra.


In [3]:
apoyo = prot[prot["rank_evo2"].notna()].copy()
apoyo["promedio_inicial_evo2"] = (apoyo["rank_inicial"] + apoyo["rank_evo2"]) / 2
print(apoyo.sort_values("promedio_inicial_evo2")[["ID", "protein_change", "rank_inicial", "rank_evo2",
                                                  "promedio_inicial_evo2"]].to_string(index=False))


    ID protein_change  rank_inicial  rank_evo2  promedio_inicial_evo2
VAR007    p.Gly247Asp             3        1.0                    2.0
VAR003    p.Arg408Trp             1        4.0                    2.5
VAR005    p.Asp338Ala             2        3.0                    2.5
VAR011    p.Thr117Ile             4        5.0                    4.5
VAR014     p.Glu76Asp             7        2.0                    4.5
VAR015     p.Tyr24Cys             6        7.0                    6.5
VAR013     p.Ile94Val             8        6.0                    7.0


In [4]:
FINAL = [
    ("VAR007", "Primer lugar de Evo2 en el consenso, ultra-rara y con un cambio de carga en el dominio catalítico."),
    ("VAR003", "Cambio químico más drástico del panel y mayor penalización en la ventana del gen, pero señal casi nula en el sitio de esa hebra."),
    ("VAR005", "Ultra-rara, pierde una carga en el dominio catalítico; tercer lugar de Evo2."),
    ("VAR014", "Ausente en la población y penalizada de forma consistente en ambas hebras; el cambio conservador la deja detrás de las tres catalíticas."),
    ("VAR011", "Ultra-rara y cerca de un sitio de splicing; Evo2 la penaliza solo en la orientación del gen."),
]

info = comp.set_index("ID")
filas = []
for pos, (vid, motivo) in enumerate(FINAL, start=1):
    ini = int(info.loc[vid, "rank_inicial"])
    dif = ini - pos
    cambio = f"Sube {dif}" if dif > 0 else f"Baja {-dif}" if dif < 0 else "Igual"
    filas.append({"Rank_final": pos, "Variante": f"{vid} {info.loc[vid, 'protein_change']}",
                  "Rank_inicial": ini, "Cambio": cambio, "Evidencia": motivo})
ranking_final = pd.DataFrame(filas)
ranking_final.to_csv("resultados/ranking_final.tsv", sep="\t", index=False)
print(ranking_final.to_string(index=False))

salen = [v for v in pd.read_csv("resultados/ranking_inicial.tsv", sep="\t")["Variante"].str.split().str[0]
         if v not in [f[0] for f in FINAL]]
print("\nSalen del Top 5 inicial:", salen)


 Rank_final           Variante  Rank_inicial Cambio                                                                                                                                Evidencia
          1 VAR007 p.Gly247Asp             3 Sube 2                                       Primer lugar de Evo2 en el consenso, ultra-rara y con un cambio de carga en el dominio catalítico.
          2 VAR003 p.Arg408Trp             1 Baja 1         Cambio químico más drástico del panel y mayor penalización en la ventana del gen, pero señal casi nula en el sitio de esa hebra.
          3 VAR005 p.Asp338Ala             2 Baja 1                                                             Ultra-rara, pierde una carga en el dominio catalítico; tercer lugar de Evo2.
          4  VAR014 p.Glu76Asp             7 Sube 3 Ausente en la población y penalizada de forma consistente en ambas hebras; el cambio conservador la deja detrás de las tres catalíticas.
          5 VAR011 p.Thr117Ile             4 Baja 1    

## 4. Completar la tabla maestra

Se llenan las columnas finales de `resultados/tabla_maestra.tsv`. Las variantes fuera del Top 5 quedan con su lugar en el orden general.


In [5]:
tm = pd.read_csv("resultados/tabla_maestra.tsv", sep="\t", dtype=str).fillna("")
pos_final = {vid: i for i, (vid, _) in enumerate(FINAL, start=1)}

LIMITES = {
    "missense": "Predicción de secuencia; sin datos de actividad enzimática ni estabilidad.",
    "delecion_in_frame": "Evo2 no evalúa indels; sin modelo estructural ni datos de actividad.",
    "sinonima": "No cambia la proteína; no se evaluó el efecto sobre el splicing.",
    "fuera_del_CDS": "Intrónica; no se evaluó el efecto sobre el splicing ni la expresión.",
}

for i, vid in enumerate(tm["variant_id"]):
    r = info.loc[vid]
    tm.loc[i, "RF3_or_structure_result"] = (texto_estructura(vid) if r["consequence"] in ("missense", "delecion_in_frame")
                                            else "No aplica: no cambia la proteína")
    tm.loc[i, "final_priority"] = str(pos_final[vid]) if vid in pos_final else "Fuera del Top 5"
    tm.loc[i, "limitations"] = LIMITES.get(r["consequence"], "")

tm.to_csv("resultados/tabla_maestra.tsv", sep="\t", index=False)
print(tm[["variant_id", "protein_change", "initial_priority", "final_priority",
          "RF3_or_structure_result", "limitations"]].to_string(index=False))


variant_id     protein_change initial_priority  final_priority          RF3_or_structure_result                                                                limitations
    VAR001          p.Asn426=               12 Fuera del Top 5 No aplica: no cambia la proteína           No cambia la proteína; no se evaluó el efecto sobre el splicing.
    VAR002          p.Tyr414=               10 Fuera del Top 5 No aplica: no cambia la proteína           No cambia la proteína; no se evaluó el efecto sobre el splicing.
    VAR003        p.Arg408Trp                1               2                        Pendiente Predicción de secuencia; sin datos de actividad enzimática ni estabilidad.
    VAR004          p.Leu385=               14 Fuera del Top 5 No aplica: no cambia la proteína           No cambia la proteína; no se evaluó el efecto sobre el splicing.
    VAR005        p.Asp338Ala                2               3                        Pendiente Predicción de secuencia; sin datos de actividad e

## 5. Conclusión

Observamos que de las 15 variantes solo ocho cambian la proteína, y que VAR007, VAR003, VAR005 y VAR011 aparecen entre las más prioritarias tanto en nuestro ranking inicial como en Evo2. A partir de esto, inferimos que VAR007 (p.Gly247Asp) es la variante con la evidencia más consistente de afectar a la PAH: es ultra-rara, cambia una glicina por un aminoácido con carga negativa dentro del dominio catalítico y es la que Evo2 penaliza más (ΔlogP −7.83). VAR003 (p.Arg408Trp) queda muy cerca, ya que tiene el cambio químico más drástico, aunque su resultado en Evo2 varía según la medida. En conclusión, VAR007 y VAR003 son nuestros mejores candidatos, y combinar varias capas de evidencia dio un mejor resultado que usar solo una. Sin embargo, todavía falta el análisis estructural para confirmar este orden, y ningún análisis midió directamente la actividad de la enzima.

## 6. Límites

- Los criterios y pesos del ranking inicial los definió el equipo.
- Los límites de dominio son aproximados, salvo el dominio ACT anotado en UniProt (residuos 36–114).
- Evo2 mide qué tan esperada es una secuencia; no mide patogenicidad, y algunas variantes dan señales distintas según la hebra o la medida.
- Evo2 no pudo evaluar la deleción VAR012.
- El ranking final puede cambiar cuando se integren los resultados de RF3 y la comparación con 6HYC.
- Ninguna capa mide actividad enzimática, estabilidad de la proteína ni splicing.

## Cierre de la sección

- **Observamos:** tres variantes del dominio catalítico (VAR007, VAR003 y VAR005) tienen evidencia convergente entre el ranking inicial y Evo2.
- **Inferimos que:** VAR007 es la candidata más consistente, seguida muy de cerca por VAR003.
- **La evidencia aporta:** frecuencia poblacional, consecuencia proteica, química del cambio, dominio y ΔlogP de Evo2.
- **Concluimos:** ranking final VAR007, VAR003, VAR005, VAR014 y VAR011, guardado en `resultados/ranking_final.tsv`.
- **Falta evidencia:** modelos RF3, comparación con 6HYC y cualquier medida experimental de actividad.
